<a href="https://colab.research.google.com/github/donovanbonner/cassandra-brain-tumor-classifier/blob/main/notebooks/cassandra_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cassandra — Brain Tumor MRI Classifier

**Authors:** Donovan Bonner, Moria El Akaya
**Course:** CS3270 Intelligent Systems, Spring 2026

## Version History
- **v1** — Baseline: 93.3% test accuracy, 0.940 macro precision. Glioma recall at 0.752.
- **v2** — In progress: adding class weighting to address glioma recall.

In [2]:
import os
import torch
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = 'donovanbonner'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [3]:
!pip install kaggle -q

!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip -q brain-tumor-mri-dataset.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 157M/157M [00:00<00:00, 224MB/s]



In [4]:
import os
for folder in os.listdir('/content/data/Training'):
    count = len(os.listdir(f'/content/data/Training/{folder}'))
    print(f"{folder}: {count} images")

for folder in os.listdir('/content/data/Testing'):
    count = len(os.listdir(f'/content/data/Testing/{folder}'))
    print(f"{folder}: {count} images")

pituitary: 1400 images
meningioma: 1400 images
glioma: 1400 images
notumor: 1400 images
pituitary: 400 images
meningioma: 400 images
glioma: 400 images
notumor: 400 images


In [13]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/data/Training', transform=train_transforms)
test_dataset  = datasets.ImageFolder('/content/data/Testing',  transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("Classes:", train_dataset.classes)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training batches: 175
Testing batches: 50


In [17]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

cassandra = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

for param in cassandra.features.parameters():
    param.requires_grad = False

for param in cassandra.features[-3:].parameters():
    param.requires_grad = True

num_features = cassandra.classifier[1].in_features
cassandra.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, 4)
)

cassandra = cassandra.to(device)

trainable_params = sum(p.numel() for p in cassandra.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in cassandra.parameters())
print(f"Total parameters:    {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

Training on: cuda
Total parameters:    4,729,600
Trainable parameters: 3,877,792
Frozen parameters:    851,808


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

EPOCHS = 15
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam([
    {'params': cassandra.features[-3:].parameters(), 'lr': 1e-5},
    {'params': cassandra.classifier.parameters(),    'lr': 1e-3}
])

print(f"Training Cassandra for {EPOCHS} epochs on {device}...\n")

for epoch in range(EPOCHS):
    cassandra.train()
    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        outputs = cassandra(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        progress_bar.set_postfix({
            'loss': f'{running_loss/(progress_bar.n+1):.3f}',
            'acc':  f'{100.*correct/total:.1f}%'
        })

    epoch_acc = 100. * correct / total
    print(f"Epoch {epoch+1} complete — Training Accuracy: {epoch_acc:.2f}%\n")

print("Cassandra training run has concluded.")

Training Cassandra for 15 epochs on cuda...



Epoch 1/15: 100%|██████████| 175/175 [00:44<00:00,  3.90it/s, loss=0.482, acc=82.0%]


Epoch 1 complete — Training Accuracy: 82.04%



Epoch 2/15: 100%|██████████| 175/175 [00:44<00:00,  3.90it/s, loss=0.304, acc=88.8%]


Epoch 2 complete — Training Accuracy: 88.79%



...

Epoch 13 complete — Training Accuracy: 96.61%



Epoch 14/15: 100%|██████████| 175/175 [00:44<00:00,  3.89it/s, loss=0.079, acc=96.9%]


Epoch 14 complete — Training Accuracy: 96.89%



Epoch 15/15: 100%|██████████| 175/175 [00:44<00:00,  3.94it/s, loss=0.079, acc=97.2%]

Epoch 15 complete — Training Accuracy: 97.25%

✅ Training complete!


In [19]:
from sklearn.metrics import classification_report
import torch

cassandra.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = cassandra(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=train_dataset.classes,
    digits=3
))

              precision    recall  f1-score   support

      glioma      0.997     0.752     0.858       400
  meningioma      0.855     0.985     0.915       400
     notumor      0.932     1.000     0.965       400
   pituitary      0.975     0.995     0.985       400

    accuracy                          0.933      1600
   macro avg      0.940     0.933     0.931      1600
weighted avg      0.940     0.933     0.931      1600



In [20]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/cassandra', exist_ok=True)

torch.save(cassandra.state_dict(), '/content/drive/MyDrive/cassandra/cassandra_v1.pth')
print("✅ Cassandra v1 saved to Drive")

Mounted at /content/drive
✅ Cassandra v1 saved to Drive
